# Phase 2: Deterministic Temperature Sensor Monitoring

This notebook ingests `artifacts/aligned_phase1_temperature.csv`, derives the run-specific 90th percentile from `artifacts/phase1_average_statistics.csv`, applies the deterministic re-sync state machine, and exports `artifacts/phase2_temperature_sensors.csv`.

Related documentation to consult while working through this notebook:
- `Distributed_Monitoring_POC_Project_Plan.md` for the canonical design and policy rules.
- `scripts/README.md` for the Phase 1/Phase 2 workflow overview.
- `instructions/PYTHON_ENV_SETUP_GUIDE.md` for environment setup details.
- `instructions/distributed_monitoring_notebook_required_changes.md` for the implementation notes that shaped the current logic.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

## Stage 1: Input Loading and Policy Resolution

This stage validates input artifacts, resolves preferred and fallback source policies, and normalizes numeric columns required by the deterministic state machine.

Run context note: repository paths resolve from `Path.cwd()` first and fall back to its parent when the notebook is executed from the `scripts` directory. If artifacts still are not found under the resolved project root, run the notebook from the expected repository root or adjust path resolution.

In [ ]:
# Stage 1: Load and normalize Phase 1 inputs for deterministic processing.
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scripts").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = resolve_repo_root()
PHASE1_ALIGNED_PATH = REPO_ROOT / "artifacts/aligned_phase1_temperature.csv"
PHASE1_STATS_PATH = REPO_ROOT / "artifacts/phase1_average_statistics.csv"
PHASE2_OUTPUT_PATH = REPO_ROOT / "artifacts/phase2_temperature_sensors.csv"
PHASE2_METRICS_PATH = REPO_ROOT / "artifacts/phase2_temperature_metrics.json"

# Preferred sources are non-imputed; fallbacks preserve compatibility with legacy artifacts.
PRIMARY_SOURCE_SERIES_NAME = "non_imputed_devices_only"
FALLBACK_SOURCE_SERIES_NAME = "all_devices_including_imputed"
PRIMARY_GLOBAL_AVG_COLUMN = "average_temperature_non_imputed_devices"
FALLBACK_GLOBAL_AVG_COLUMN = "average_temperature_all_devices"

SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
GLOBAL_AVG_COLUMN = PRIMARY_GLOBAL_AVG_COLUMN

def require_input_files(paths):
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Required input not found: {path}")

def resolve_global_average_column(df):
    # Prefer non-imputed aggregate; if absent, use all-device aggregate.
    if PRIMARY_GLOBAL_AVG_COLUMN in df.columns:
        return PRIMARY_GLOBAL_AVG_COLUMN
    if FALLBACK_GLOBAL_AVG_COLUMN in df.columns:
        return FALLBACK_GLOBAL_AVG_COLUMN
    raise ValueError(
        "Could not find a valid global average column in aligned phase1 data"
    )

def select_p90_row(stats_frame):
    # Keep deterministic fallback order for p90 provenance.
    row = stats_frame.loc[stats_frame["series_name"] == PRIMARY_SOURCE_SERIES_NAME]
    if row.empty:
        row = stats_frame.loc[stats_frame["series_name"] == FALLBACK_SOURCE_SERIES_NAME]
    if row.empty:
        raise ValueError(
            "Could not find p90 source row in phase1_average_statistics.csv"
        )
    return row

def find_sensor_columns(df):
    # Per-sensor columns follow *_temperature; exclude aggregate averages.
    cols = sorted(
        [
            c
            for c in df.columns
            if c.endswith("_temperature")
            and c not in [PRIMARY_GLOBAL_AVG_COLUMN, FALLBACK_GLOBAL_AVG_COLUMN]
        ]
    )
    if not cols:
        raise ValueError("No per-sensor temperature columns found in aligned phase1 data")
    return cols

require_input_files([PHASE1_ALIGNED_PATH, PHASE1_STATS_PATH])

phase1_df = pd.read_csv(PHASE1_ALIGNED_PATH)
stats_df = pd.read_csv(PHASE1_STATS_PATH)

if phase1_df.empty:
    raise ValueError("aligned_phase1_temperature.csv has no rows")

GLOBAL_AVG_COLUMN = resolve_global_average_column(phase1_df)
SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
p90_row = select_p90_row(stats_df)
# Capture the actual source used so output metadata records p90 provenance.
if p90_row.iloc[0]["series_name"] != PRIMARY_SOURCE_SERIES_NAME:
    SOURCE_SERIES_NAME = FALLBACK_SOURCE_SERIES_NAME

p90_threshold = float(p90_row.iloc[0]["p90"])
sensor_cols = find_sensor_columns(phase1_df)

phase1_df = phase1_df.sort_values("bucket_epoch", kind="mergesort").reset_index(drop=True)
phase1_df[GLOBAL_AVG_COLUMN] = pd.to_numeric(phase1_df[GLOBAL_AVG_COLUMN], errors="coerce")
for col in sensor_cols:
    phase1_df[col] = pd.to_numeric(phase1_df[col], errors="coerce")

first_xbar = float(phase1_df.loc[0, GLOBAL_AVG_COLUMN])
if np.isnan(first_xbar):
    non_null_global = phase1_df[GLOBAL_AVG_COLUMN].dropna()
    if non_null_global.empty:
        raise ValueError("Global average column has no numeric values")
    first_xbar = float(non_null_global.iloc[0])

print(
    json.dumps(
        {
            "repo_root": str(REPO_ROOT),
            "rows": int(len(phase1_df)),
            "sensor_count": int(len(sensor_cols)),
            "p90_threshold": p90_threshold,
            "global_avg_column": GLOBAL_AVG_COLUMN,
            "source_series_name": SOURCE_SERIES_NAME,
        },
        indent=2,
    )
)

## Stage 2: Deterministic Re-sync State Machine

This stage applies row-by-row state transitions, computes local trigger requests, tracks re-sync causes, and writes the detailed Phase 2 operational dataset.

State transition contract per row:
- Read the `entry_*` state carried into this minute.
- Read the `observed_*` values for this minute bucket.
- Compute `event_*` outcomes using only `entry_*` references and margins for local trigger evaluation.
- Commit the resulting `exit_*` state at the end of the iteration; that state becomes the next row's `entry_*` state.

Row invariants:
- All `entry_*` fields are the state used to evaluate this row.
- All `observed_*` fields are raw values from the current minute bucket.
- All `event_*` fields are outcomes produced during this minute.
- All `exit_*` fields describe the state that leaves this row and becomes active next row.

Transition summary note:
- `transition_summary` is a compact pipe-delimited summary of the row's event and exit-state changes, placed early in the output for audit readability.

In [ ]:
def round_or_blank(value):
    return "" if np.isnan(value) else round(float(value), 4)

def compute_resync_update(row, sensor_columns, global_average, fallback_xbar, threshold):
    # Re-sync recomputes a shared margin and refreshes available sensor references.
    if np.isnan(global_average):
        computed_xbar = float(fallback_xbar)
    else:
        computed_xbar = float(global_average)
    computed_delta = float(threshold - computed_xbar)

    refreshed_refs = {}
    for sensor_column in sensor_columns:
        sensor_value = row[sensor_column]
        if not np.isnan(sensor_value):
            refreshed_refs[sensor_column] = float(sensor_value)

    refreshed_margins = {sensor_column: computed_delta for sensor_column in sensor_columns}
    return computed_xbar, computed_delta, refreshed_refs, refreshed_margins

def build_transition_summary(
    event_resync_consumed_from_prior_row,
    event_any_sensor_requested_resync,
    event_resync_triggered_by_local_violation,
    event_exit_delta_negative,
    exit_resync_scheduled_next_row,
):
    parts = []
    if event_resync_consumed_from_prior_row == 1:
        parts.append("consumed_prior_resync")
    if event_any_sensor_requested_resync == 1:
        parts.append("local_request")
    if event_resync_triggered_by_local_violation == 1:
        parts.append("local_resync")
    if event_exit_delta_negative == 1:
        parts.append("exit_delta_negative")
    if exit_resync_scheduled_next_row == 1:
        parts.append("scheduled_next_resync")
    if not parts:
        return "steady_state"
    return "|".join(parts)

records = []
sensor_count = len(sensor_cols)

# State at row t: baseline global average, baseline delta, and per-sensor reference/margin.
prior_xbar = first_xbar
prior_delta_global = p90_threshold - prior_xbar
sensor_reference_values = {
    sensor_col: (
        float(phase1_df.loc[0, sensor_col])
        if not np.isnan(phase1_df.loc[0, sensor_col])
        else np.nan
    )
    for sensor_col in sensor_cols
}
local_margin_values = {sensor_col: float(prior_delta_global) for sensor_col in sensor_cols}

# This flag is set by row t and consumed by row t+1.
resync_due_next_row = 0

for _, row in phase1_df.iterrows():
    global_avg = row[GLOBAL_AVG_COLUMN]

    entry_xbar_t0 = float(prior_xbar)
    entry_delta_global = float(prior_delta_global)

    event_resync_consumed_from_prior_row = int(resync_due_next_row == 1)
    exit_resync_scheduled_next_row = 0
    event_resync_performed = 0
    event_resync_triggered_by_local_violation = 0
    event_resync_reason = ""

    trigger_message_count = 0
    request_message_count = 0
    response_message_count = 0
    broadcast_message_count = 0

    sensor_fields = {}
    triggering_sensor_names = []

    next_prior_xbar = float(prior_xbar)
    next_prior_delta_global = float(prior_delta_global)
    next_sensor_reference_values = dict(sensor_reference_values)
    next_local_margin_values = dict(local_margin_values)

    forced_resync_row = event_resync_consumed_from_prior_row == 1
    if forced_resync_row:
        (
            computed_xbar,
            computed_delta,
            refreshed_refs,
            refreshed_margins,
        ) = compute_resync_update(
            row=row,
            sensor_columns=sensor_cols,
            global_average=global_avg,
            fallback_xbar=prior_xbar,
            threshold=p90_threshold,
        )

        next_sensor_reference_values.update(refreshed_refs)
        next_local_margin_values = refreshed_margins
        next_prior_xbar = computed_xbar
        next_prior_delta_global = computed_delta

        event_resync_performed = 1
        event_resync_reason = "negative_delta_from_prior_row"

        request_message_count = sensor_count
        response_message_count = sensor_count
        broadcast_message_count = sensor_count

        # Forced rows consume scheduled re-sync and must not emit local trigger requests.
        for sensor_col in sensor_cols:
            sensor_name = sensor_col.replace("_temperature", "")
            observed_sensor_col = f"observed_{sensor_name}_temperature"
            sensor_ref_col = f"entry_{sensor_name}_reference_value"
            local_dev_col = f"event_{sensor_name}_local_deviation"
            local_margin_col = f"entry_{sensor_name}_local_margin"
            trigger_col = f"event_{sensor_name}_resync_requested"

            sensor_val = row[sensor_col]
            ref_val = sensor_reference_values.get(sensor_col, np.nan)
            local_margin = local_margin_values.get(sensor_col, np.nan)

            sensor_fields[observed_sensor_col] = round_or_blank(sensor_val)
            sensor_fields[sensor_ref_col] = round_or_blank(ref_val)
            sensor_fields[local_margin_col] = round_or_blank(local_margin)
            sensor_fields[local_dev_col] = ""
            sensor_fields[trigger_col] = 0

            # A zero-deviation sensor cannot be marked as locally triggering due to negative margin.
            if (
                not np.isnan(sensor_val)
                and not np.isnan(ref_val)
                and not np.isnan(local_margin)
                and local_margin < 0
                and np.isclose(float(sensor_val) - float(ref_val), 0.0)
            ):
                assert sensor_fields[trigger_col] == 0

        event_resync_request_count = 0
        event_any_sensor_requested_resync = 0
    else:
        # Trigger checks intentionally use entry-state references/margins for this row.
        for sensor_col in sensor_cols:
            sensor_name = sensor_col.replace("_temperature", "")
            observed_sensor_col = f"observed_{sensor_name}_temperature"
            sensor_ref_col = f"entry_{sensor_name}_reference_value"
            local_dev_col = f"event_{sensor_name}_local_deviation"
            local_margin_col = f"entry_{sensor_name}_local_margin"
            trigger_col = f"event_{sensor_name}_resync_requested"

            sensor_val = row[sensor_col]
            ref_val = sensor_reference_values.get(sensor_col, np.nan)
            local_margin = local_margin_values.get(sensor_col, np.nan)

            sensor_fields[observed_sensor_col] = round_or_blank(sensor_val)
            sensor_fields[sensor_ref_col] = round_or_blank(ref_val)
            sensor_fields[local_margin_col] = round_or_blank(local_margin)

            if np.isnan(sensor_val) or np.isnan(ref_val):
                sensor_fields[local_dev_col] = ""
                sensor_fields[trigger_col] = 0
                continue

            local_deviation = float(sensor_val) - float(ref_val)
            sensor_fields[local_dev_col] = round(local_deviation, 4)

            if np.isnan(local_margin):
                trigger_bit = 0
            else:
                # Warming-only policy: cooling deviations do not trigger re-sync.
                trigger_bit = int(local_deviation >= float(local_margin))

            sensor_fields[trigger_col] = trigger_bit
            if trigger_bit == 1:
                triggering_sensor_names.append(sensor_name)

        event_resync_request_count = int(len(triggering_sensor_names))
        event_any_sensor_requested_resync = int(event_resync_request_count > 0)

        if event_resync_request_count > 0:
            trigger_message_count = event_resync_request_count

            (
                computed_xbar,
                computed_delta,
                refreshed_refs,
                refreshed_margins,
            ) = compute_resync_update(
                row=row,
                sensor_columns=sensor_cols,
                global_average=global_avg,
                fallback_xbar=prior_xbar,
                threshold=p90_threshold,
            )

            next_sensor_reference_values.update(refreshed_refs)
            next_local_margin_values = refreshed_margins
            next_prior_xbar = computed_xbar
            next_prior_delta_global = computed_delta

            event_resync_performed = 1
            event_resync_triggered_by_local_violation = 1
            event_resync_reason = "local_constraint_violation"

            request_message_count = sensor_count
            response_message_count = sensor_count
            broadcast_message_count = sensor_count

    # Canonical boundary rule for global violation is >= p90 threshold.
    global_violation = int(
        (not np.isnan(global_avg)) and (float(global_avg) >= float(p90_threshold))
    )

    # If exit delta turns negative at row i, force re-sync application at row i+1.
    event_exit_delta_negative = int(next_prior_delta_global < 0)
    if event_exit_delta_negative == 1:
        exit_resync_scheduled_next_row = 1

    # Invariants for forced-row processing.
    if event_resync_consumed_from_prior_row == 1:
        assert trigger_message_count == 0
        assert event_resync_triggered_by_local_violation == 0
        assert event_any_sensor_requested_resync == 0

    # Message semantics: trigger=originating sensors, request/response/broadcast=network fanout.
    total_message_count = int(
        trigger_message_count
        + request_message_count
        + response_message_count
        + broadcast_message_count
    )

    transition_summary = build_transition_summary(
        event_resync_consumed_from_prior_row=event_resync_consumed_from_prior_row,
        event_any_sensor_requested_resync=event_any_sensor_requested_resync,
        event_resync_triggered_by_local_violation=event_resync_triggered_by_local_violation,
        event_exit_delta_negative=event_exit_delta_negative,
        exit_resync_scheduled_next_row=exit_resync_scheduled_next_row,
    )

    out = {
        "bucket_epoch": int(row["bucket_epoch"]),
        "bucket_time_utc": row["bucket_time_utc"],
        "transition_summary": transition_summary,
        "p90_threshold_used": round(float(p90_threshold), 4),
        "observed_global_average": round_or_blank(global_avg),
        "entry_xbar_t0": round(entry_xbar_t0, 4),
        "entry_delta_global": round(entry_delta_global, 4),
        "exit_delta_global": round(float(next_prior_delta_global), 4),
        "global_violation": global_violation,
        "event_triggering_sensor_names": "|".join(triggering_sensor_names),
        "event_resync_request_count": event_resync_request_count,
        "event_any_sensor_requested_resync": event_any_sensor_requested_resync,
        "event_resync_consumed_from_prior_row": event_resync_consumed_from_prior_row,
        "event_resync_triggered_by_local_violation": event_resync_triggered_by_local_violation,
        "event_resync_performed": event_resync_performed,
        "event_resync_reason": event_resync_reason,
        "event_exit_delta_negative": event_exit_delta_negative,
        "exit_resync_scheduled_next_row": exit_resync_scheduled_next_row,
        "exit_xbar_t0_if_resync": "" if event_resync_performed == 0 else round(float(next_prior_xbar), 4),
        "exit_delta_global_if_resync": "" if event_resync_performed == 0 else round(float(next_prior_delta_global), 4),
        "trigger_message_count": int(trigger_message_count),
        "request_message_count": int(request_message_count),
        "response_message_count": int(response_message_count),
        "broadcast_message_count": int(broadcast_message_count),
        "total_message_count": int(total_message_count),
    }

    out.update(sensor_fields)
    records.append(out)

    prior_xbar = float(next_prior_xbar)
    prior_delta_global = float(next_prior_delta_global)
    sensor_reference_values = dict(next_sensor_reference_values)
    local_margin_values = dict(next_local_margin_values)
    resync_due_next_row = int(exit_resync_scheduled_next_row)

phase2_df = pd.DataFrame.from_records(records)
PHASE2_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
phase2_df.to_csv(PHASE2_OUTPUT_PATH, index=False)

print(f"Wrote {len(phase2_df)} rows to {PHASE2_OUTPUT_PATH}")

## Stage 3: Metrics and Communication Diagnostics

This stage derives classification-style diagnostics, communication cost comparisons, and detection delay summaries from the generated Phase 2 dataset.

Interpretation note: lag values are bucket offsets from the start of each contiguous violation window, not absolute row indices or timestamps.

In [ ]:
def compute_confusion_counts(actual_series, predicted_series):
    tp = int(((actual_series == 1) & (predicted_series == 1)).sum())
    fn = int(((actual_series == 1) & (predicted_series == 0)).sum())
    fp = int(((actual_series == 0) & (predicted_series == 1)).sum())
    tn = int(((actual_series == 0) & (predicted_series == 0)).sum())
    return tp, fn, fp, tn

def compute_window_lags(actual_series, predicted_series):
    # Lag unit is buckets since the start of each contiguous actual-positive window.
    lags = []
    in_window = False
    window_start = None

    for idx, val in enumerate(actual_series.tolist()):
        if val == 1 and not in_window:
            in_window = True
            window_start = idx
        elif val == 0 and in_window:
            window_end = idx - 1
            segment = predicted_series.iloc[window_start : window_end + 1]
            hit_positions = np.where(segment.to_numpy() == 1)[0]
            if len(hit_positions) > 0:
                lags.append(int(hit_positions[0]))
            else:
                lags.append(None)
            in_window = False

    if in_window and window_start is not None:
        segment = predicted_series.iloc[window_start:]
        hit_positions = np.where(segment.to_numpy() == 1)[0]
        if len(hit_positions) > 0:
            lags.append(int(hit_positions[0]))
        else:
            lags.append(None)

    return lags

def round4(value):
    return round(float(value), 4)

actual = phase2_df["global_violation"].astype(int)
predicted = phase2_df["event_any_sensor_requested_resync"].astype(int)

tp, fn, fp, tn = compute_confusion_counts(actual, predicted)

recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

sensor_count = len(sensor_cols)
baseline_sensor_messages = len(phase2_df) * sensor_count

distributed_total_messages = int(phase2_df["total_message_count"].sum())
distributed_trigger_messages = int(phase2_df["trigger_message_count"].sum())
distributed_request_messages = int(phase2_df["request_message_count"].sum())
distributed_response_messages = int(phase2_df["response_message_count"].sum())
distributed_broadcast_messages = int(phase2_df["broadcast_message_count"].sum())

# Primary ratio uses total distributed traffic; legacy ratio preserves old trigger-only baseline comparison.
communication_reduction_ratio_primary = (
    1.0 - (distributed_total_messages / baseline_sensor_messages)
    if baseline_sensor_messages > 0
    else np.nan
)
communication_reduction_ratio_legacy = (
    1.0 - (distributed_trigger_messages / baseline_sensor_messages)
    if baseline_sensor_messages > 0
    else np.nan
)

window_lags = compute_window_lags(actual, predicted)
valid_lags = [x for x in window_lags if x is not None]

# Metrics sections document rule definitions, quality diagnostics, and communication cost.
metrics = {
    "rows": int(len(phase2_df)),
    "sensor_count": int(sensor_count),
    "p90_threshold": round4(p90_threshold),
    "boundary_policy": {
        "canonical_rule": "violated_if_observed_global_average_gte_p90",
        "variant_policy": "strict_gt_available_for_diagnostic_sensitivity_only",
    },
    "trigger_policy": {
        "local_trigger_rule": "event_{sensor}_local_deviation >= entry_{sensor}_local_margin",
        "predicted_positive_rule": "event_any_sensor_requested_resync == 1",
    },
    "confusion": {
        "tp": tp,
        "fn": fn,
        "fp": fp,
        "tn": tn,
    },
    "recall_diagnostic": None if np.isnan(recall) else round4(recall),
    "precision_diagnostic": None if np.isnan(precision) else round4(precision),
    "communication": {
        "distributed_total_messages": distributed_total_messages,
        "distributed_trigger_messages": distributed_trigger_messages,
        "distributed_request_messages": distributed_request_messages,
        "distributed_response_messages": distributed_response_messages,
        "distributed_broadcast_messages": distributed_broadcast_messages,
        "centralized_sensor_messages": int(baseline_sensor_messages),
        "communication_reduction_ratio_primary": None
        if np.isnan(communication_reduction_ratio_primary)
        else round4(communication_reduction_ratio_primary),
        "communication_reduction_ratio_legacy": None
        if np.isnan(communication_reduction_ratio_legacy)
        else round4(communication_reduction_ratio_legacy),
    },
    "resync_events": {
        "resync_performed_count": int(phase2_df["event_resync_performed"].sum()),
        "negative_delta_detected_count": int(phase2_df["event_exit_delta_negative"].sum()),
        "scheduled_from_prior_count": int(phase2_df["event_resync_consumed_from_prior_row"].sum()),
    },
    "detection_delay_by_window": {
        "window_lags_in_buckets": window_lags,
        "mean_lag": None if not valid_lags else round4(np.mean(valid_lags)),
        "max_lag": None if not valid_lags else int(max(valid_lags)),
    },
}

with PHASE2_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))
print(f"Wrote metrics to {PHASE2_METRICS_PATH}")